# 29 · Аудит честности: активные строки vs «раннее предупреждение»

P(событие ≤ 24ч) и топ-K на holdout могут выигрывать от того, что в ТЕКУЩЕМ
6ч-бакете уже есть событие (задымление/неисправность/тревога). Тогда это не
«раннее предупреждение», а «продолжение начавшегося». Проверяем по трём задачам:

1. Важна ли в модели колонка текущего бакета (`серьёзных`, `задым_подтв_об_б` и т.п.);
2. PR-AUC(24ч/7д/30д) и топ-K на ВСЕХ строках и на «чистых» (нет события
   в текущем бакете, ни по каналу, ни по объекту);
3. Сколько из событий holdout падает на «чистые» строки.

Прод-смысл: если топ-K = «объект уже горит/задымлён в этом бакете», диспетчеру
нужна иная формулировка карточки (продолжение инцидента, а не предсказание).


In [1]:
%matplotlib inline
import pathlib
import sys

_HERE = pathlib.Path.cwd()
RESEARCH = _HERE.parent if _HERE.name == "notebooks" else _HERE
if str(RESEARCH) not in sys.path:
    sys.path.insert(0, str(RESEARCH))

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier

import data_utils as du
import tte_pipeline as tte
from tte_experiments import pr_at_horizon, predict_survival_discrete

DATASET = du.find_dataset_dir()
MODELS = DATASET.parent / "models"


In [2]:
# 1. Топ-фичи fire: есть ли текущие событийные колонки?
s = pd.read_parquet(DATASET / "tte_subjects_fire.parquet")
train, _, _ = tte.split_subjects(s)
X_cols = tte.subject_features(train)
m = CatBoostClassifier(task_type="CPU")
m.load_model(MODELS / "tte_fire_discrete_hazard.cbm")
imp = pd.Series(m.get_feature_importance(type="PredictionValuesChange"),
                index=list(m.feature_names_))
print("ТОП-25 фич fire:")
print(imp.sort_values(ascending=False).head(25).round(3).to_string())


ТОП-25 фич fire:
час_шага                           25.634
день_недели_шага                   13.354
шаг                                 6.120
доля_горизонта                      5.050
дни_с_посл_старта                   3.301
каналов_задымлений_об_б_сум_30д     2.611
дни_с_посл_неиспр                   2.565
событий_об_б_сум_30д                2.512
ид_объект_code                      2.266
каналов_задымлений_об_б             2.036
задымлений_об_б_сум_30д             1.943
месяц_шага                          1.503
день_года                           1.430
тревог_дым_об_б_сум_30д             1.353
неисправностей_об_б_сум_48ч         1.175
неисправностей_об_б                 1.142
каналов_задымлений_об_б_сум_12ч     0.992
каналов_активных_об_б               0.948
событий_сум_30д                     0.945
каналов_тревог_дым_об_б_сум_7д      0.904
задым_подтв_об_б_сум_30д            0.899
неисправностей_об_б_сум_24ч         0.835
неисправностей_об_б_сум_30д         0.811
тревог_об_б_сум_4

In [3]:
# 2. Активность в текущем бакете: определения
#    A) по каналу: в текущем бакете у КАНАЛА событие (серьёзных/тревог/неисправностей>0)
#    B) по объекту: у ОБЪЕКТА подтверждено (задым_подтв_об_б>0)
TASKS = {
    "fire":  ("серьёзных", "задым_подтв_об_б"),
    "access":("тревог", "тревог"),
    "sensor":("неисправностей", "неисправностей"),
}

rows = []
for task, (ev, obj_ev) in TASKS.items():
    m = CatBoostClassifier(task_type="CPU")
    m.load_model(MODELS / f"tte_{task}_discrete_hazard.cbm")
    s = pd.read_parquet(DATASET / f"tte_subjects_{task}.parquet")
    train, _, holdout = tte.split_subjects(s)
    X_cols = tte.subject_features(train)
    ho = tte.sample_holdout(holdout, 20_000, seed=777)
    S = predict_survival_discrete(m, ho, X_cols)
    step24 = int(round(1.0 * tte.STEPS_PER_DAY)) - 1
    p24 = 1.0 - S[:, step24]
    risk30 = 1.0 - S[:, -1]
    y24 = ((ho["event_flag"] == 1) & (ho["obs_days"] <= 1.01)).to_numpy(int)
    y30 = ((ho["event_flag"] == 1) & (ho["obs_days"] <= 30.01)).to_numpy(int)
    active_ch = ho[ev].to_numpy() > 0
    active_obj = ho[obj_ev].to_numpy() > 0
    active = active_ch | active_obj
    clean = ~active
    pr_all = pr_at_horizon(y24, p24)
    pr_clean = pr_at_horizon(y24[clean], p24[clean])
    pr30_all = pr_at_horizon(y30, risk30)
    pr30_clean = pr_at_horizon(y30[clean], risk30[clean])
    order = np.argsort(-p24)
    top100_act = int(active[order[:100]].sum())
    order30 = np.argsort(-risk30)
    top200_act30 = int(active[order30[:200]].sum())
    rows.append({
        "task": task,
        "n": len(ho),
        "active%": active.mean() * 100,
        "PR-AUC24 ALL": pr_all["pr_auc"],
        "PR-AUC24 CLEAN": pr_clean["pr_auc"],
        "PR-AUC30 ALL": pr30_all["pr_auc"],
        "PR-AUC30 CLEAN": pr30_clean["pr_auc"],
        "top100 active": top100_act,
        "top200@risk30 active": top200_act30,
        "events24 total": int(y24.sum()),
        "events24 clean": int(y24[clean].sum()),
        "events24 active": int(y24[active].sum()),
    })
print(pd.DataFrame(rows).round(4).to_string(index=False))


  task     n  active%  PR-AUC24 ALL  PR-AUC24 CLEAN  PR-AUC30 ALL  PR-AUC30 CLEAN  top100 active  top200@risk30 active  events24 total  events24 clean  events24 active
  fire 19999  13.9457        0.4206          0.0315        0.4864          0.4435            100                    86             306             130              176
access 19999   9.6655        0.0304          0.0307        0.6190          0.3706              0                   169             202             202                0
sensor 19999   1.0451        0.0159          0.0178        0.0959          0.0734             18                    68              61              61                0


In [4]:
# 3. fire детально: PR-AUC по горизонтам на чистых строках
s = pd.read_parquet(DATASET / "tte_subjects_fire.parquet")
train, _, holdout = tte.split_subjects(s)
X_cols = tte.subject_features(train)
m = CatBoostClassifier(task_type="CPU")
m.load_model(MODELS / "tte_fire_discrete_hazard.cbm")
ho = tte.sample_holdout(holdout, 20_000, seed=777)
S = predict_survival_discrete(m, ho, X_cols)
active = (ho["серьёзных"].to_numpy() > 0) | (ho["задым_подтв_об_б"].to_numpy() > 0)
clean = ~active
lines = []
for h, label in ((1.0, "24ч"), (7.0, "7д"), (30.0, "30д")):
    yh = ((ho["event_flag"] == 1) & (ho["obs_days"] <= h + 0.01)).to_numpy(int)
    ph = 1.0 - S[:, int(round(h * tte.STEPS_PER_DAY)) - 1]
    pa = pr_at_horizon(yh, ph)
    pc = pr_at_horizon(yh[clean], ph[clean])
    lines.append("%s ALL: PR-AUC=%.3f (база %.2f%%) | CLEAN: PR-AUC=%.3f (база %.2f%%)" % (
        label, pa["pr_auc"], pa["base"] * 100, pc["pr_auc"], pc["base"] * 100))
print("\n".join(lines))
print("\nАктивных строк: %.2f%%" % (active.mean() * 100))


24ч ALL: PR-AUC=0.421 (база 1.53%) | CLEAN: PR-AUC=0.032 (база 0.76%)
7д ALL: PR-AUC=0.308 (база 7.66%) | CLEAN: PR-AUC=0.199 (база 6.41%)
30д ALL: PR-AUC=0.486 (база 19.58%) | CLEAN: PR-AUC=0.443 (база 18.46%)

Активных строк: 13.95%
